# Lab 2: Embedding Generation and Vector Search with LangChain (Free Version)

## Introduction

Welcome to Lab 2! In this lab, we'll build upon the document processing pipeline from Lab 1 and explore embedding generation and vector search using completely free tools.

### Learning Objectives
- Generate embeddings using free Hugging Face models
- Store embeddings in ChromaDB (free vector database)
- Perform similarity search and retrieval
- Compare different embedding models and search strategies

## 1. Environment Setup

In [1]:
# Install required packages
!pip install langchain sentence-transformers pypdf beautifulsoup4 requests langchain_community

In [13]:
!pip install langchain-chroma

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
# Import necessary libraries
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import requests
import time
import numpy as np

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 2. Load and Process Documents

Let's start by loading some documents using the techniques from Lab 1.

In [4]:
# Load a sample document (Wikipedia page about Machine Learning)
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Machine_learning")
docs = loader.load()

print(f"📄 Loaded {len(docs)} documents")
print(f"📝 Content preview: {docs[0].page_content[:200]}...")

📄 Loaded 1 documents
📝 Content preview: 



Machine learning - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout ...


In [5]:
# Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

splits = text_splitter.split_documents(docs)
print(f"📚 Split into {len(splits)} chunks")
print(f"📏 Average chunk size: {sum(len(doc.page_content) for doc in splits) / len(splits):.0f} characters")

📚 Split into 175 chunks
📏 Average chunk size: 751 characters


In [19]:
type(splits)

list

## 3. Understanding Embeddings

Before we generate embeddings, let's understand what they are and why they're important for RAG systems.

In [6]:
# Let's see what embeddings look like
sample_texts = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "The weather is nice today.",
    "Artificial intelligence and machine learning are related fields."
]

print("📝 Sample texts:")
for i, text in enumerate(sample_texts, 1):
    print(f"   {i}. {text}")

📝 Sample texts:
   1. Machine learning is a subset of artificial intelligence.
   2. Deep learning uses neural networks with multiple layers.
   3. The weather is nice today.
   4. Artificial intelligence and machine learning are related fields.


## 4. Generating Embeddings with Hugging Face

We'll use Hugging Face's sentence transformers to generate embeddings. These models are free and run locally!

In [7]:
# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",  # Fast and efficient model
    model_kwargs={'device': 'cpu'},  # Use CPU (works on all machines)
    encode_kwargs={'normalize_embeddings': True}  # Normalize for better similarity search
)

print("✅ Embedding model loaded!")
print(f"📊 Model: all-MiniLM-L6-v2")
print(f"💾 Device: CPU")

/var/folders/sc/r0vk6bp90xz8qtd3lqtpsrzc0000gn/T/ipykernel_31988/4250306260.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/opt/homebrew/anaconda3/envs/audioenv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


✅ Embedding model loaded!
📊 Model: all-MiniLM-L6-v2
💾 Device: CPU


In [8]:
# Generate embeddings for our sample texts
sample_embeddings = embeddings.embed_documents(sample_texts)

print(f"🔢 Generated {len(sample_embeddings)} embeddings")
print(f"📐 Embedding dimension: {len(sample_embeddings[0])}")
print(f"\n📊 First embedding (first 10 values):")
print([f"{x:.4f}" for x in sample_embeddings[0][:10]])

🔢 Generated 4 embeddings
📐 Embedding dimension: 384

📊 First embedding (first 10 values):
['-0.0235', '-0.0106', '0.0719', '0.0304', '0.0286', '-0.0388', '-0.0282', '-0.0163', '-0.0576', '-0.0034']


In [9]:
# Let's compute similarity between texts manually
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("🔍 Similarity matrix:")
print("     Text 1  Text 2  Text 3  Text 4")
for i, text_i in enumerate(sample_texts):
    similarities = []
    for j, text_j in enumerate(sample_texts):
        sim = cosine_similarity(sample_embeddings[i], sample_embeddings[j])
        similarities.append(f"{sim:.3f}")
    print(f"Text {i+1}: {' '.join(similarities)}")

print("\n💡 Notice: Texts 1, 2, and 4 (about AI/ML) are more similar to each other than to text 3 (about weather)!")

🔍 Similarity matrix:
     Text 1  Text 2  Text 3  Text 4
Text 1: 1.000 0.489 -0.035 0.800
Text 2: 0.489 1.000 -0.040 0.487
Text 3: -0.035 -0.040 1.000 -0.052
Text 4: 0.800 0.487 -0.052 1.000

💡 Notice: Texts 1, 2, and 4 (about AI/ML) are more similar to each other than to text 3 (about weather)!


## 5. Creating a Vector Store with ChromaDB

Now let's store our document embeddings in ChromaDB, a free vector database that's perfect for learning and development.

In [20]:
pip install faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 2.8 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [21]:
from langchain_community.vectorstores import FAISS
# Create a vector store from our document splits
print("🔄 Creating vector store...")
start_time = time.time()

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)

# Optional: Save to disk (FAISS uses different method)
vectorstore.save_local("./faiss_index")
end_time = time.time()
print(f"✅ Vector store created in {end_time - start_time:.2f} seconds")
print(f"📚 Stored {len(splits)} document chunks")
print(f"💾 Persisted to: ./faiss_index")

🔄 Creating vector store...
✅ Vector store created in 3.19 seconds
📚 Stored 175 document chunks
💾 Persisted to: ./chroma_db


## 6. Performing Similarity Search

Now comes the exciting part - searching for relevant documents based on a query!

In [22]:
# Perform a basic similarity search
query = "What is supervised learning?"
docs = vectorstore.similarity_search(query, k=3)

print(f"🔍 Query: '{query}'")
print(f"📄 Found {len(docs)} relevant documents:\n")

for i, doc in enumerate(docs, 1):
    print(f"📋 Document {i}:")
    print(f"   Content: {doc.page_content[:200]}...")
    print(f"   Source: {doc.metadata.get('source', 'Unknown')}")
    print()

🔍 Query: 'What is supervised learning?'
📄 Found 3 relevant documents:

📋 Document 1:
   Content: Supervised learning[edit]
Main article: Supervised learning
A support-vector machine is a supervised learning model that divides the data into regions separated by a linear boundary. Here, the linear ...
   Source: https://en.wikipedia.org/wiki/Machine_learning

📋 Document 2:
   Content: Supervised learning algorithms build a mathematical model of a set of data that contains both the inputs and the desired outputs.[45] The data, known as training data, consists of a set of training ex...
   Source: https://en.wikipedia.org/wiki/Machine_learning

📋 Document 3:
   Content: Approaches[edit]


In supervised learning, the training data is labelled with the expected answers, while in unsupervised learning, the model identifies patterns or structures in unlabelled data.
Mach...
   Source: https://en.wikipedia.org/wiki/Machine_learning



In [23]:
# Search with scores to see how similar the documents are
docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"🔍 Query: '{query}'")
print(f"📊 Results with similarity scores:\n")

for i, (doc, score) in enumerate(docs_with_scores, 1):
    print(f"📋 Document {i} (Score: {score:.4f}):")
    print(f"   {doc.page_content[:150]}...")
    print()

🔍 Query: 'What is supervised learning?'
📊 Results with similarity scores:

📋 Document 1 (Score: 0.5613):
   Supervised learning[edit]
Main article: Supervised learning
A support-vector machine is a supervised learning model that divides the data into regions...

📋 Document 2 (Score: 0.6615):
   Supervised learning algorithms build a mathematical model of a set of data that contains both the inputs and the desired outputs.[45] The data, known ...

📋 Document 3 (Score: 0.6791):
   Approaches[edit]


In supervised learning, the training data is labelled with the expected answers, while in unsupervised learning, the model identifi...



## 7. Trying Different Queries

Let's test our vector store with various types of queries to see how well it retrieves relevant information.

In [24]:
# Test different types of queries
test_queries = [
    "neural networks and deep learning",
    "training data and algorithms",
    "artificial intelligence applications",
    "overfitting and model validation"
]

for query in test_queries:
    print(f"🔍 Query: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    
    for i, doc in enumerate(docs, 1):
        print(f"   📄 Result {i}: {doc.page_content[:100]}...")
    print()

🔍 Query: 'neural networks and deep learning'
   📄 Result 1: The original goal of the ANN approach was to solve problems in the same way that a human brain would...
   📄 Result 2: Artificial neural networks[edit]
Main article: Artificial neural networkSee also: Deep learning
An a...

🔍 Query: 'training data and algorithms'
   📄 Result 1: Supervised learning algorithms build a mathematical model of a set of data that contains both the in...
   📄 Result 2: Glossary
Glossary
vte
Machine learning (ML) is a field of study in artificial intelligence concerned...

🔍 Query: 'artificial intelligence applications'
   📄 Result 1: The original goal of the ANN approach was to solve problems in the same way that a human brain would...
   📄 Result 2: Approaches
Machine learning
Symbolic
Deep learning
Bayesian networks
Evolutionary algorithms
Hybrid ...

🔍 Query: 'overfitting and model validation'
   📄 Result 1: Model assessments[edit]
Classification of machine learning models can be validated by accur

## 8. Comparing Different Embedding Models

Let's compare different free embedding models to see how they perform.

In [25]:
# Test different embedding models
models_to_test = [
    "all-MiniLM-L6-v2",      # Fast and lightweight
    "all-mpnet-base-v2",     # Better quality, larger
    "paraphrase-MiniLM-L6-v2"  # Good for paraphrase detection
]

model_results = {}

for model_name in models_to_test:
    print(f"🧪 Testing model: {model_name}")
    
    # Initialize model
    test_embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cpu'}
    )
    
    # Time the embedding generation
    start_time = time.time()
    test_embedding = test_embeddings.embed_query("test query")
    end_time = time.time()
    
    model_results[model_name] = {
        'dimension': len(test_embedding),
        'time': end_time - start_time
    }
    
    print(f"   📐 Dimension: {len(test_embedding)}")
    print(f"   ⏱️  Time: {end_time - start_time:.3f} seconds")
    print()

print("📊 Model Comparison Summary:")
for model, results in model_results.items():
    print(f"   {model}: {results['dimension']}D, {results['time']:.3f}s")

🧪 Testing model: all-MiniLM-L6-v2
   📐 Dimension: 384
   ⏱️  Time: 0.021 seconds

🧪 Testing model: all-mpnet-base-v2
   📐 Dimension: 768
   ⏱️  Time: 0.742 seconds

🧪 Testing model: paraphrase-MiniLM-L6-v2
   📐 Dimension: 384
   ⏱️  Time: 0.058 seconds

📊 Model Comparison Summary:
   all-MiniLM-L6-v2: 384D, 0.021s
   all-mpnet-base-v2: 768D, 0.742s
   paraphrase-MiniLM-L6-v2: 384D, 0.058s


## 9. Advanced Search Techniques

Let's explore some advanced search techniques available in ChromaDB.

In [26]:
# Maximal Marginal Relevance (MMR) search
# This helps get diverse results, not just the most similar ones
query = "machine learning algorithms"

print(f"🔍 Query: '{query}'")
print("\n📄 Regular similarity search:")
regular_docs = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(regular_docs, 1):
    print(f"   {i}. {doc.page_content[:80]}...")

print("\n🎯 MMR search (more diverse results):")
mmr_docs = vectorstore.max_marginal_relevance_search(query, k=3, fetch_k=10)
for i, doc in enumerate(mmr_docs, 1):
    print(f"   {i}. {doc.page_content[:80]}...")

🔍 Query: 'machine learning algorithms'

📄 Regular similarity search:
   1. Glossary
Glossary
vte
Machine learning (ML) is a field of study in artificial in...
   2. ^ "Machine Learning Algorithms". GeeksforGeeks. 17 August 2023. Retrieved 3 Sept...
   3. Supervised learning[edit]
Main article: Supervised learning
A support-vector mac...

🎯 MMR search (more diverse results):
   1. Glossary
Glossary
vte
Machine learning (ML) is a field of study in artificial in...
   2. Learning classifier systems (LCS) are a family of rule-based machine learning al...
   3. Supervised learning[edit]
Main article: Supervised learning
A support-vector mac...


## 10. Working with Metadata

Let's add some metadata to our documents and use it for filtering.

In [27]:
# Create documents with enhanced metadata
enhanced_docs = []
for i, doc in enumerate(splits[:10]):  # Use first 10 chunks
    enhanced_metadata = doc.metadata.copy()
    enhanced_metadata.update({
        'topic': 'machine_learning',
        'difficulty': 'intermediate' if i % 2 == 0 else 'beginner',
        'chunk_id': i,
        'section': 'introduction' if i < 5 else 'advanced'
    })
    
    enhanced_doc = Document(
        page_content=doc.page_content,
        metadata=enhanced_metadata
    )
    enhanced_docs.append(enhanced_doc)

# Create a new vector store with enhanced metadata
enhanced_vectorstore = FAISS.from_documents(
    documents=enhanced_docs,
    embedding=embeddings
)
enhanced_vectorstore.save_local("./faiss_enhanced_index")
print(f"✅ Created enhanced vector store with {len(enhanced_docs)} documents")
print("📋 Sample metadata:")
for key, value in enhanced_docs[0].metadata.items():
    print(f"   {key}: {value}")

✅ Created enhanced vector store with 10 documents
📋 Sample metadata:
   source: https://en.wikipedia.org/wiki/Machine_learning
   title: Machine learning - Wikipedia
   language: en
   topic: machine_learning
   difficulty: intermediate
   chunk_id: 0
   section: introduction


In [28]:
# Search with metadata filtering
query = "learning algorithms"

print(f"🔍 Query: '{query}'")
print("\n📄 All results:")
all_results = enhanced_vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(all_results, 1):
    difficulty = doc.metadata.get('difficulty', 'unknown')
    section = doc.metadata.get('section', 'unknown')
    print(f"   {i}. [{difficulty}, {section}] {doc.page_content[:60]}...")

print("\n🎯 Filtered results (beginner level only):")
# Note: ChromaDB filtering syntax may vary by version
try:
    filtered_results = enhanced_vectorstore.similarity_search(
        query, 
        k=3,
        filter={"difficulty": "beginner"}
    )
    for i, doc in enumerate(filtered_results, 1):
        difficulty = doc.metadata.get('difficulty', 'unknown')
        section = doc.metadata.get('section', 'unknown')
        print(f"   {i}. [{difficulty}, {section}] {doc.page_content[:60]}...")
except Exception as e:
    print(f"   Filtering not available in this ChromaDB version: {e}")

🔍 Query: 'learning algorithms'

📄 All results:
   1. [beginner, advanced] Glossary
Glossary
vte
Machine learning (ML) is a field of st...
   2. [beginner, introduction] 2.3
Data mining








2.4
Generalization








2.5
Stati...
   3. [intermediate, introduction] 5.7
Gaussian processes








5.8
Genetic algorithms






...

🎯 Filtered results (beginner level only):
   1. [beginner, advanced] Glossary
Glossary
vte
Machine learning (ML) is a field of st...
   2. [beginner, introduction] 2.3
Data mining








2.4
Generalization








2.5
Stati...
   3. [beginner, advanced] Part of a series onMachine learningand data mining
Paradigms...


## 11. Exercises

Now it's your turn to practice! Complete the following exercises.

### Exercise 1: Multi-Document Vector Store

Load documents from 3 different sources and create a combined vector store.

In [ ]:
# Exercise 1: Your code here
# TODO:
# 1. Load documents from 3 different Wikipedia pages
# 2. Split them into chunks
# 3. Create a combined vector store
# 4. Test search across all documents

# Your code here:


### Exercise 2: Embedding Model Comparison

Compare the search results from different embedding models on the same query.

In [ ]:
# Exercise 2: Your code here
# TODO:
# 1. Create vector stores with 2 different embedding models
# 2. Search the same query in both
# 3. Compare and analyze the results

# Your code here:


### Exercise 3: Custom Similarity Function

Implement a custom function to find the most similar documents manually.

In [ ]:
# Exercise 3: Your code here
# TODO:
# 1. Get embeddings for all documents
# 2. Get embedding for a query
# 3. Calculate similarities manually
# 4. Return top-k most similar documents

# Your code here:


### Exercise 4: Persistent Vector Store

Create a vector store, save it, and load it back in a new session.

In [ ]:
# Exercise 4: Your code here
# TODO:
# 1. Create a vector store with persist_directory
# 2. Save it using .persist()
# 3. Load it back using Chroma(persist_directory=...)
# 4. Test that it works correctly

# Your code here:


## Solutions

Here are the solutions to the exercises. Try to complete them yourself first!

In [31]:
# Solution 1: Multi-Document Vector Store
urls = [
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Computer_vision"
]

all_docs = []
for url in urls:
    loader = WebBaseLoader(url)
    docs = loader.load()
    # Add source identifier to metadata
    for doc in docs:
        doc.metadata['topic'] = url.split('/')[-1].replace('_', ' ')
    all_docs.extend(docs)
    print(f"✅ Loaded: {url.split('/')[-1]}")

# Split all documents
all_splits = text_splitter.split_documents(all_docs)

# Create combined vector store
# Create combined vector store
multi_vectorstore = FAISS.from_documents(
    documents=all_splits,
    embedding=embeddings
)
multi_vectorstore.save_local("./multi_faiss_index")

# Test cross-document search
query = "neural networks for image recognition"
results = multi_vectorstore.similarity_search(query, k=3)

print(f"\n🔍 Query: '{query}'")
for i, doc in enumerate(results, 1):
    topic = doc.metadata.get('topic', 'unknown')
    print(f"   {i}. [{topic}] {doc.page_content[:80]}...")

✅ Loaded: Deep_learning
✅ Loaded: Natural_language_processing
✅ Loaded: Computer_vision

🔍 Query: 'neural networks for image recognition'
   1. [Deep learning] Artificial neural networks (ANNs) or connectionist systems are computing systems...
   2. [Deep learning] Neural networks[edit]
Main article: Artificial neural network...
   3. [Deep learning] ^ Cireşan, Dan Claudiu; Meier, Ueli; Gambardella, Luca Maria; Schmidhuber, Jürge...


In [32]:
# Solution 2: Embedding Model Comparison
models = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]
query = "supervised learning algorithms"

for model_name in models:
    print(f"\n🧪 Testing {model_name}:")
    
    # Create embeddings and vector store
    test_embeddings = HuggingFaceEmbeddings(model_name=model_name)
    test_vectorstore = FAISS.from_documents(
        documents=splits[:20],  # Use subset for speed
        embedding=test_embeddings
    )
    
    # Search
    results = test_vectorstore.similarity_search_with_score(query, k=2)
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"   {i}. Score: {score:.4f} - {doc.page_content[:60]}...")


🧪 Testing all-MiniLM-L6-v2:
   1. Score: 0.9205 - Glossary
Glossary
vte
Machine learning (ML) is a field of st...
   2. Score: 0.9207 - Current Supervised Learning Algorithms have objectives of cl...

🧪 Testing all-mpnet-base-v2:
   1. Score: 0.6775 - Part of a series onMachine learningand data mining
Paradigms...
   2. Score: 0.8041 - 2.3
Data mining








2.4
Generalization








2.5
Stati...


In [33]:
# Solution 3: Custom Similarity Function
def custom_similarity_search(query, documents, embeddings_model, k=3):
    # Get embeddings for all documents
    doc_texts = [doc.page_content for doc in documents]
    doc_embeddings = embeddings_model.embed_documents(doc_texts)
    
    # Get query embedding
    query_embedding = embeddings_model.embed_query(query)
    
    # Calculate similarities
    similarities = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_embedding, doc_emb)
        similarities.append((sim, i, documents[i]))
    
    # Sort by similarity and return top-k
    similarities.sort(reverse=True)
    return similarities[:k]

# Test custom function
query = "machine learning models"
custom_results = custom_similarity_search(query, splits[:10], embeddings, k=3)

print(f"🔍 Custom search for: '{query}'")
for i, (sim, idx, doc) in enumerate(custom_results, 1):
    print(f"   {i}. Similarity: {sim:.4f} - {doc.page_content[:60]}...")

🔍 Custom search for: 'machine learning models'
   1. Similarity: 0.5925 - Glossary
Glossary
vte
Machine learning (ML) is a field of st...
   2. Similarity: 0.5058 - 5.7
Gaussian processes








5.8
Genetic algorithms






...
   3. Similarity: 0.4918 - 2.3
Data mining








2.4
Generalization








2.5
Stati...


In [36]:
# Solution 4: Persistent Vector Store
# Create and save vector store
# Create and save vector store
persistent_vectorstore = FAISS.from_documents(
    documents=splits[:15],
    embedding=embeddings
)
persistent_vectorstore.save_local("./persistent_faiss_index_v2")
print("✅ Vector store saved")

# Load vector store back
loaded_vectorstore = FAISS.load_local(
    "./persistent_faiss_index",
    embeddings, allow_dangerous_deserialization=True
)

# Test that it works
test_query = "artificial intelligence"
results = loaded_vectorstore.similarity_search(test_query, k=2)

print(f"✅ Loaded vector store works!")
print(f"🔍 Test query: '{test_query}'")
for i, doc in enumerate(results, 1):
    print(f"   {i}. {doc.page_content[:60]}...")

✅ Vector store saved
✅ Loaded vector store works!
🔍 Test query: 'artificial intelligence'
   1. Learning with humans
Active learning
Crowdsourcing
Human-in-...
   2. Approaches
Machine learning
Symbolic
Deep learning
Bayesian ...


## Summary

Congratulations! You've completed Lab 2. Here's what you've learned:

✅ **Embeddings**: How to generate semantic embeddings using free Hugging Face models

✅ **Vector Stores**: How to store and manage embeddings using ChromaDB

✅ **Similarity Search**: How to find relevant documents based on semantic similarity

✅ **Model Comparison**: How different embedding models affect search quality and performance

✅ **Advanced Techniques**: MMR search, metadata filtering, and persistence

### Key Insights

1. **Embeddings capture semantic meaning** - Similar concepts cluster together in vector space
2. **Model choice matters** - Larger models generally provide better quality but are slower
3. **ChromaDB is powerful** - Provides persistence, metadata filtering, and advanced search
4. **Free tools are capable** - You can build production-quality search with open-source tools

### Next Steps

In Lab 3, we'll combine everything to build a complete RAG chatbot:
- Use our vector store as a retriever
- Connect it to a local language model (Ollama)
- Create conversational chains with memory
- Build a complete question-answering system

Great job! 🎉